In [ ]:
# @title Set ONLINE = False if running locally
ONLINE = True

In [ ]:
#@title Imports

import numpy as np
import pandas as pd
import math
from numba import njit
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import gdown
import os

In [ ]:
# Download training and submission data
if ONLINE:
    url = 'https://drive.google.com/file/d/1lDEtOq644qlaw_vh0i5c3ChVfrHJiuMc/view?usp=sharing'
    output = 'data_train.csv'
    gdown.download(url, output, fuzzy=True)
    url = 'https://drive.google.com/file/d/1KMiakfzTWYr4QJU9S5DC9e5KIo_8bb24/view?usp=sharing'
    output = 'sampleSubmission.csv'
    gdown.download(url, output, fuzzy=True)

In [ ]:
# Define the clipping range based on your rating scale
MIN_RATING = 1.0  # Minimum possible rating
MAX_RATING = 5.0  # Maximum possible rating

# Load data
data_pd = pd.read_csv('data_train.csv')  # Read training data from CSV file

def extract_users_items_predictions(data_pd):
    """
    Extracts user IDs, item (movie) IDs, and predictions from the dataset.

    Args:
    - data_pd (DataFrame): DataFrame containing columns 'Id' and 'Prediction'.

    Returns:
    - users (ndarray): Array of user indices.
    - movies (ndarray): Array of movie indices.
    - predictions (ndarray): Array of prediction values.
    """
    users, movies = [
        np.squeeze(arr) for arr in np.split(
            data_pd.Id.str.extract('r(\d+)_c(\d+)').values.astype(int) - 1, 2, axis=-1
        )
    ]
    predictions = data_pd.Prediction.values
    return users, movies, predictions

def prepare_data(data_pd, train_size=0.9, random_state=42):
    """
    Splits the dataset into training and testing sets and extracts users, items, and predictions.

    Args:
    - data_pd (DataFrame): The complete dataset.
    - train_size (float): Proportion of the dataset to include in the train split.
    - random_state (int): Random seed for reproducibility.

    Returns:
    - train_users (ndarray): Training set user indices.
    - train_movies (ndarray): Training set movie indices.
    - train_predictions (ndarray): Training set predictions.
    - test_users (ndarray): Testing set user indices.
    - test_movies (ndarray): Testing set movie indices.
    - test_predictions (ndarray): Testing set predictions.
    """
    # Split the data into training and testing sets
    train_pd, test_pd = train_test_split(data_pd, train_size=train_size, random_state=random_state)

    # Extract user, movie, and prediction data from the training and testing sets
    train_users, train_movies, train_predictions = extract_users_items_predictions(train_pd)
    test_users, test_movies, test_predictions = extract_users_items_predictions(test_pd)

    return train_users, train_movies, train_predictions, test_users, test_movies, test_predictions

def initialize_matrices(number_of_users, number_of_movies, train_predictions, train_users, train_movies):
    """
    Initializes the data and mask matrices for the training process.

    Args:
    - number_of_users (int): Total number of users.
    - number_of_movies (int): Total number of movies.
    - train_predictions (ndarray): Predictions from the training data.
    - train_users (ndarray): User indices from the training data.
    - train_movies (ndarray): Movie indices from the training data.

    Returns:
    - data (ndarray): Matrix of user-movie ratings initialized with mean predictions.
    - mask (ndarray): Binary mask indicating observed ratings in the data matrix.
    """
    # Initialize the data matrix with the mean of the training predictions
    data = np.full((number_of_users, number_of_movies), np.mean(train_predictions))

    # Initialize the mask matrix with zeros
    mask = np.zeros((number_of_users, number_of_movies))

    # Populate the data and mask matrices with training predictions
    for user, movie, pred in zip(train_users, train_movies, train_predictions):
        data[user][movie] = pred
        mask[user][movie] = 1

    return data, mask

def rmse(x, y):
    """
    Calculates the Root Mean Square Error (RMSE) between two arrays.

    Args:
    - x (ndarray): Array of predicted values.
    - y (ndarray): Array of true values.

    Returns:
    - float: The RMSE between x and y.
    """
    return math.sqrt(mean_squared_error(x, y))

def initialize_factors(number_of_users, number_of_movies, latent_factors, implicit_factors):
    """
    Initializes the user and item latent factor matrices for collaborative filtering.

    Args:
    - number_of_users (int): Total number of users.
    - number_of_movies (int): Total number of movies.
    - latent_factors (int): Number of latent factors for explicit ratings.
    - implicit_factors (int): Number of latent factors for implicit interactions.

    Returns:
    - pu (ndarray): User latent factors matrix.
    - qi (ndarray): Item latent factors matrix.
    - pu_imp (ndarray): User implicit latent factors matrix.
    - qi_imp (ndarray): Item implicit latent factors matrix.
    """
    # Initialize user and item latent factor matrices with small random values
    pu = np.random.normal(scale=1./latent_factors, size=(number_of_users, latent_factors))
    qi = np.random.normal(scale=1./latent_factors, size=(number_of_movies, latent_factors))

    # Initialize user and item implicit latent factor matrices with small random values
    pu_imp = np.random.normal(scale=1./latent_factors, size=(number_of_users, implicit_factors))
    qi_imp = np.random.normal(scale=1./latent_factors, size=(number_of_movies, implicit_factors))

    return pu, qi, pu_imp, qi_imp

def compute_biases(number_of_users, number_of_movies, train_users, train_movies, train_predictions):
    """
    Computes biases for users and items based on training data.

    Args:
    - number_of_users (int): Total number of users.
    - number_of_movies (int): Total number of movies.
    - train_users (ndarray): User indices from the training data.
    - train_movies (ndarray): Movie indices from the training data.
    - train_predictions (ndarray): Predictions from the training data.

    Returns:
    - bu (ndarray): User bias array.
    - bi (ndarray): Item bias array.
    - global_mean (float): Global mean rating.
    """
    # Calculate the global mean rating from the training predictions
    global_mean = np.mean(train_predictions)

    # Initialize user and item average ratings and counts
    user_avg_rating = np.zeros(number_of_users)
    item_avg_rating = np.zeros(number_of_movies)
    user_count = np.zeros(number_of_users)
    item_count = np.zeros(number_of_movies)

    # Accumulate ratings and counts for users and items
    for user, movie, rating in zip(train_users, train_movies, train_predictions):
        user_avg_rating[user] += rating
        user_count[user] += 1
        item_avg_rating[movie] += rating
        item_count[movie] += 1

    # Compute average ratings, handling division by zero
    user_avg_rating /= np.where(user_count == 0, 1, user_count)
    item_avg_rating /= np.where(item_count == 0, 1, item_count)

    # Calculate user and item biases as deviations from the global mean
    bu = user_avg_rating - global_mean
    bi = item_avg_rating - global_mean

    return bu, bi, global_mean

def initialize_adam_moments(pu, qi, bu, bi, pu_imp, qi_imp):
    """
    Initializes Adam optimization moments for user/item factors and biases.

    Args:
    - pu (ndarray): User latent factors matrix.
    - qi (ndarray): Item latent factors matrix.
    - bu (ndarray): User bias array.
    - bi (ndarray): Item bias array.
    - pu_imp (ndarray): User implicit latent factors matrix.
    - qi_imp (ndarray): Item implicit latent factors matrix.

    Returns:
    - m_pu, v_pu, m_qi, v_qi, m_bu, v_bu, m_bi, v_bi, m_pu_imp, v_pu_imp, m_qi_imp, v_qi_imp (tuples of ndarrays):
      Moment vectors for Adam optimization.
    """
    # Initialize the first moment (mean) and second moment (uncentered variance) vectors for Adam optimization
    m_pu, v_pu = np.zeros_like(pu), np.zeros_like(pu)
    m_qi, v_qi = np.zeros_like(qi), np.zeros_like(qi)
    m_bu, v_bu = np.zeros_like(bu), np.zeros_like(bu)
    m_bi, v_bi = np.zeros_like(bi), np.zeros_like(bi)
    m_pu_imp, v_pu_imp = np.zeros_like(pu_imp), np.zeros_like(pu_imp)
    m_qi_imp, v_qi_imp = np.zeros_like(qi_imp), np.zeros_like(qi_imp)

    return m_pu, v_pu, m_qi, v_qi, m_bu, v_bu, m_bi, v_bi, m_pu_imp, v_pu_imp, m_qi_imp, v_qi_imp

@njit
def update_learning_rate(initial_lr, epoch, decay_rate, stop_decay):
    """
    Updates the learning rate using exponential decay, with a floor on the decay.

    Args:
    - initial_lr (float): Initial learning rate.
    - epoch (int): Current epoch number.
    - decay_rate (float): Rate of decay.
    - stop_decay (float): Minimum learning rate threshold.

    Returns:
    - float: Updated learning rate.
    """
    # Calculate the decayed learning rate
    curr = initial_lr * (decay_rate ** epoch)

    # Ensure learning rate does not drop below the stop decay threshold
    return max(curr, stop_decay)

@njit
def adam_update(param, grad, m, v, t, learning_rate, beta1, beta2, epsilon):
    """
    Performs a single update step using the Adam optimization algorithm.

    Args:
    - param (ndarray): Current parameter array to be updated.
    - grad (ndarray): Gradient of the loss with respect to the parameter.
    - m (ndarray): First moment vector (mean).
    - v (ndarray): Second moment vector (variance).
    - t (int): Current time step (epoch number).
    - learning_rate (float): Learning rate.
    - beta1 (float): Exponential decay rate for the first moment.
    - beta2 (float): Exponential decay rate for the second moment.
    - epsilon (float): Small constant for numerical stability.

    Returns:
    - param (ndarray): Updated parameter array.
    - m (ndarray): Updated first moment vector.
    - v (ndarray): Updated second moment vector.
    """
    # Update biased first moment estimate
    m = beta1 * m + (1 - beta1) * grad

    # Update biased second raw moment estimate
    v = beta2 * v + (1 - beta2) * (grad ** 2)

    # Compute bias-corrected first moment estimate
    m_hat = m / (1 - beta1 ** t)

    # Compute bias-corrected second raw moment estimate
    v_hat = v / (1 - beta2 ** t)

    # Update parameters with computed learning rate
    param += learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)

    return param, m, v

def extract_prediction_from_full_matrix(reconstructed_matrix, users, movies):
    """
    Extracts predictions from a reconstructed user-item rating matrix.

    Args:
    - reconstructed_matrix (ndarray): Full matrix with predicted ratings.
    - users (ndarray): Array of user indices.
    - movies (ndarray): Array of movie indices.

    Returns:
    - predictions (ndarray): Extracted predicted ratings.
    """
    # Ensure that the lengths of users and movies arrays are equal
    assert len(users) == len(movies), "users-movies combinations specified should have equal length"

    # Initialize an array for predictions
    predictions = np.zeros(len(users))

    # Extract predictions for each user-movie pair
    for i, (user, movie) in enumerate(zip(users, movies)):
        predictions[i] = reconstructed_matrix[user][movie]

    return predictions

def find_optimal_svd_rmse(data, test_users, test_movies, test_predictions):
    """
    Finds the optimal number of singular values for SVD by minimizing RMSE.

    Args:
    - data (ndarray): User-item ratings matrix.
    - test_users (ndarray): Array of test user indices.
    - test_movies (ndarray): Array of test movie indices.
    - test_predictions (ndarray): Array of test prediction values.

    Returns:
    - best_rmse (float): The best RMSE obtained with optimal number of singular values.
    """
    # Initialize bounds for number of singular values to test
    min_singular_values = 1
    max_singular_values = min(data.shape[0], data.shape[1])

    # Initialize best parameters
    best_k = min_singular_values
    best_rmse = float('inf')

    # Iterate over possible numbers of singular values
    for k_singular_values in range(min_singular_values, max_singular_values):
        # Perform Singular Value Decomposition
        U, s, Vt = np.linalg.svd(data, full_matrices=False)

        # Reconstruct the matrix using the top k singular values
        S = np.zeros((data.shape[1], data.shape[1]))
        S[:k_singular_values, :k_singular_values] = np.diag(s[:k_singular_values])

        # Calculate reconstructed matrix
        reconstructed_matrix = U.dot(S).dot(Vt)

        # Extract predictions for test users and movies
        predictions = extract_prediction_from_full_matrix(reconstructed_matrix, test_users, test_movies)

        # Compute RMSE for current reconstruction
        current_rmse = rmse(predictions, test_predictions)

        # Update best RMSE and number of singular values if current RMSE is better
        if current_rmse < best_rmse:
            best_rmse = current_rmse
            best_k = k_singular_values

    # Output the optimal number of singular values and corresponding RMSE
    print("Optimal number of singular values: {}".format(best_k))
    print("Best RMSE using SVD is: {:.4f}".format(best_rmse))

    return best_rmse

@njit
def compute_rmse(true_ratings, predicted_ratings):
    """
    Computes the Root Mean Square Error (RMSE) between true and predicted ratings.

    Args:
    - true_ratings (ndarray): Array of true ratings.
    - predicted_ratings (ndarray): Array of predicted ratings.

    Returns:
    - float: RMSE value.
    """
    # Compute error between true and predicted ratings
    error = true_ratings - predicted_ratings

    # Return RMSE value
    return np.sqrt(np.mean(error ** 2))

@njit
def train_model(
    pu, qi, bu, bi, pu_imp, qi_imp,
    m_pu, v_pu, m_qi, v_qi, m_bu, v_bu, m_bi, v_bi,
    m_pu_imp, v_pu_imp, m_qi_imp, v_qi_imp,
    initial_learning_rate, regularization, num_epochs, decay_rate, stop_decay,
    train_users, train_movies, train_predictions, batch_size,
    beta1, beta2, epsilon):
    """
    Trains the collaborative filtering model using stochastic gradient descent with Adam optimization.

    Args:
    - pu (ndarray): User latent factors matrix.
    - qi (ndarray): Item latent factors matrix.
    - bu (ndarray): User bias array.
    - bi (ndarray): Item bias array.
    - pu_imp (ndarray): User implicit latent factors matrix.
    - qi_imp (ndarray): Item implicit latent factors matrix.
    - m_pu, v_pu, m_qi, v_qi, m_bu, v_bu, m_bi, v_bi, m_pu_imp, v_pu_imp, m_qi_imp, v_qi_imp (tuples of ndarrays):
      Moment vectors for Adam optimization.
    - initial_learning_rate (float): Initial learning rate.
    - regularization (float): Regularization parameter.
    - num_epochs (int): Number of training epochs.
    - decay_rate (float): Learning rate decay rate.
    - stop_decay (float): Minimum learning rate threshold.
    - train_users (ndarray): User indices from the training data.
    - train_movies (ndarray): Movie indices from the training data.
    - train_predictions (ndarray): Predictions from the training data.
    - batch_size (int): Batch size for stochastic gradient descent.
    - beta1 (float): Exponential decay rate for the first moment.
    - beta2 (float): Exponential decay rate for the second moment.
    - epsilon (float): Small constant for numerical stability.

    Returns:
    - Updated user/item factors and biases.
    """
    # Initialize variables for early stopping
    best_rmse = 1e10  # Set a high initial best RMSE
    patience = 20     # Maximum number of epochs to wait for improvement
    patience_counter = 0  # Counter for epochs without improvement

    # Begin training over the specified number of epochs
    for epoch in range(1, num_epochs + 1):
        # Update learning rate with decay
        learning_rate = update_learning_rate(initial_learning_rate, epoch, decay_rate, stop_decay)

        # Process data in mini-batches
        for start in range(0, len(train_users), batch_size):
            end = min(start + batch_size, len(train_users))
            batch_users = train_users[start:end]
            batch_movies = train_movies[start:end]
            batch_ratings = train_predictions[start:end]

            # Update model parameters for each example in the batch
            for user, movie, rating in zip(batch_users, batch_movies, batch_ratings):
                # Predict the rating for the current user-movie pair
                prediction = (
                    global_mean + bu[user] + bi[movie] +
                    np.dot(pu[user], qi[movie]) +
                    np.dot(pu_imp[user], qi_imp[movie])
                )

                # Calculate prediction error
                error = rating - prediction

                # Compute gradients for biases and latent factors
                grad_bu = error - regularization * bu[user]
                grad_bi = error - regularization * bi[movie]
                grad_pu = error * qi[movie] - regularization * pu[user]
                grad_qi = error * pu[user] - regularization * qi[movie]
                grad_pu_imp = error * qi_imp[movie] - regularization * pu_imp[user]
                grad_qi_imp = error * pu_imp[user] - regularization * qi_imp[movie]

                # Apply Adam updates to the model parameters
                bu[user], m_bu[user], v_bu[user] = adam_update(bu[user], grad_bu, m_bu[user], v_bu[user], epoch, learning_rate, beta1, beta2, epsilon)
                bi[movie], m_bi[movie], v_bi[movie] = adam_update(bi[movie], grad_bi, m_bi[movie], v_bi[movie], epoch, learning_rate, beta1, beta2, epsilon)
                pu[user], m_pu[user], v_pu[user] = adam_update(pu[user], grad_pu, m_pu[user], v_pu[user], epoch, learning_rate, beta1, beta2, epsilon)
                qi[movie], m_qi[movie], v_qi[movie] = adam_update(qi[movie], grad_qi, m_qi[movie], v_qi[movie], epoch, learning_rate, beta1, beta2, epsilon)
                pu_imp[user], m_pu_imp[user], v_pu_imp[user] = adam_update(pu_imp[user], grad_pu_imp, m_pu_imp[user], v_pu_imp[user], epoch, learning_rate, beta1, beta2, epsilon)
                qi_imp[movie], m_qi_imp[movie], v_qi_imp[movie] = adam_update(qi_imp[movie], grad_qi_imp, m_qi_imp[movie], v_qi_imp[movie], epoch, learning_rate, beta1, beta2, epsilon)

        # Calculate training RMSE after processing all batches
        train_predictions_svd = (
            global_mean + bu[train_users] + bi[train_movies] +
            np.sum(pu[train_users] * qi[train_movies], axis=1) +
            np.sum(pu_imp[train_users] * qi_imp[train_movies], axis=1)
        )
        train_rmse = compute_rmse(train_predictions, train_predictions_svd)

        # Check for improvement in RMSE
        if train_rmse < best_rmse:
            best_rmse = train_rmse
            patience_counter = 0  # Reset patience counter
        else:
            patience_counter += 1  # Increment patience counter

        # Implement early stopping if no improvement
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    return pu, qi, bu, bi, pu_imp, qi_imp


In [ ]:
# Define parameters
number_of_users, number_of_movies = 10000, 1000
latent_factors = 14
implicit_factors = 200
initial_learning_rate = 0.05
regularization = 0.1
num_epochs = 250
decay_rate = 0.96
stop_decay = 0
batch_size = 128
beta1, beta2 = 0.8, 0.999
epsilon = 1e-8

# Prepare data
train_users, train_movies, train_predictions, test_users, test_movies, test_predictions = prepare_data(data_pd)

# Initialize matrices and biases
data, mask = initialize_matrices(number_of_users, number_of_movies, train_predictions, train_users, train_movies)
pu, qi, pu_imp, qi_imp = initialize_factors(number_of_users, number_of_movies, latent_factors, implicit_factors)
bu, bi, global_mean = compute_biases(number_of_users, number_of_movies, train_users, train_movies, train_predictions)
m_pu, v_pu, m_qi, v_qi, m_bu, v_bu, m_bi, v_bi, m_pu_imp, v_pu_imp, m_qi_imp, v_qi_imp = initialize_adam_moments(pu, qi, bu, bi, pu_imp, qi_imp)

# Find optimal rmse of "normal" svd
best_rmse = find_optimal_svd_rmse(data, test_users, test_movies, test_predictions)

# Train model
pu, qi, bu, bi, pu_imp, qi_imp = train_model(
    pu, qi, bu, bi, pu_imp, qi_imp, m_pu, v_pu, m_qi, v_qi, m_bu, v_bu, m_bi, v_bi,
    m_pu_imp, v_pu_imp, m_qi_imp, v_qi_imp, initial_learning_rate, regularization,
    num_epochs, decay_rate, stop_decay, train_users, train_movies, train_predictions,
    batch_size, beta1, beta2, epsilon
)


In [ ]:
# Test predictions
svd_test_predictions = global_mean + bu[test_users] + bi[test_movies] + np.sum(pu[test_users] * qi[test_movies], axis=1) + np.sum(pu_imp[test_users] * qi_imp[test_movies], axis=1)

# **Clip the test predictions to the specified rating range**
svd_test_predictions_clipped = np.clip(svd_test_predictions, MIN_RATING, MAX_RATING)

# Evaluate RMSE using clipped predictions
test_rmse = rmse(test_predictions, svd_test_predictions_clipped)
print(f'Test RMSE using Improved SVD with Implicit Feedback and Adam: {test_rmse:.4f}')


In [ ]:
# Calculate SVD train predictions
svd_train_predictions = global_mean + bu[train_users] + bi[train_movies] + np.sum(pu[train_users] * qi[train_movies], axis=1) + np.sum(pu_imp[train_users] * qi_imp[train_movies], axis=1)

# **Clip the train predictions to the specified rating range**
svd_train_predictions_clipped = np.clip(svd_train_predictions, MIN_RATING, MAX_RATING)

# Calculate residuals on train set
train_residuals = train_predictions - svd_train_predictions_clipped

# Prepare data for LightGBM
X_train_lgb = np.column_stack((train_users, train_movies))
y_train_lgb = train_residuals

# Create LightGBM dataset
lgb_train_data = lgb.Dataset(X_train_lgb, label=y_train_lgb)

# Set LightGBM parameters
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.25,
    'num_leaves': 6,
    'feature_fraction': 0.9,
    'verbose': -1
}

# Train LightGBM model on residuals
lgb_model = lgb.train(params, lgb_train_data, num_boost_round=100)

# Prepare test data for LightGBM
X_test_lgb = np.column_stack((test_users, test_movies))

# Predict residuals on test set
lgb_residuals = lgb_model.predict(X_test_lgb)

# Combine SVD predictions and LightGBM residuals on test set
combined_test_predictions = svd_test_predictions_clipped + lgb_residuals

# **Clip the final combined predictions to the specified rating range**
combined_test_predictions_clipped = np.clip(combined_test_predictions, MIN_RATING, MAX_RATING)

# Evaluate the final model
final_rmse = rmse(test_predictions, combined_test_predictions_clipped)
print(f'Final RMSE: {final_rmse:.4f}')


In [ ]:
# Load sample submission file
sample_submission = pd.read_csv('sampleSubmission.csv')

# Extract user and item IDs from the 'Id' column
sample_submission['user_id'] = sample_submission['Id'].apply(lambda x: int(x.split('_')[0][1:]) - 1)
sample_submission['item_id'] = sample_submission['Id'].apply(lambda x: int(x.split('_')[1][1:]) - 1)

# Predict the ratings using the trained model
user_ids = sample_submission['user_id'].values
item_ids = sample_submission['item_id'].values

# Predict using the SVD model
svd_submission_predictions = (
    global_mean
    + bu[user_ids]
    + bi[item_ids]
    + np.sum(pu[user_ids] * qi[item_ids], axis=1)
    + np.sum(pu_imp[user_ids] * qi_imp[item_ids], axis=1)
)

# **Clip the SVD submission predictions to the specified rating range**
svd_submission_predictions_clipped = np.clip(svd_submission_predictions, MIN_RATING, MAX_RATING)

# Prepare data for LightGBM
X_lgb = np.column_stack((user_ids, item_ids))

# Predict residuals using LightGBM model
lgb_residuals_submission = lgb_model.predict(X_lgb)

# Combine SVD predictions and LightGBM residuals
submission_predictions = svd_submission_predictions_clipped + lgb_residuals_submission

# **Clip the final submission predictions to the specified rating range**
submission_predictions_clipped = np.clip(submission_predictions, MIN_RATING, MAX_RATING)

# Save predictions to .npz files
predictions_path = 'predictions'
if not os.path.exists(predictions_path):
    os.mkdir(predictions_path)

# Save the submission predictions
np.savez_compressed(
    os.path.join(predictions_path, 'svd_predictions.npz'),
    test=combined_test_predictions_clipped,
    challenge=submission_predictions_clipped
)

# Assign predictions to the sample submission file
sample_submission['Prediction'] = submission_predictions_clipped

csv_path = 'csvs'
if not os.path.exists(csv_path):
    os.mkdir(csv_path)

# Save final submission to CSV
sample_submission[['Id', 'Prediction']].to_csv(os.path.join(csv_path, 'submission_svd.csv'), index=False)